In [ ]:
#r "nuget: FsLexYacc.Runtime, 11.3.0"
#r "Scanning/bin/Debug/net9.0/Scanning.dll"
open System
open System.IO
open LectureScanning
// Run python FSharp/tools/build.py from the repository root first.

The arithmetic scanner from the PLY tutorial is translated to a typed FsLex scanner.

## A Tokenizer for Numbers and the Arithmetical Operators

FsLex compiles `Scanning/Arithmetic.fsl` into F# source. `Scanning.fsproj` pins FsLexYacc 11.3.0; no Python binding is used.

The token values form a discriminated union. `Number` carries an integer, `Symbol` a character, and `Illegal` supports the original recover-and-continue demonstration.

In [ ]:
// Defined in Scanning/Types.fs:
let tokenCases = ["Number of int"; "Symbol of char"; "Illegal of char"; "End"]
tokenCases

A `.fsl` rule contains a regular expression and an F# action. The syntax uses quoted characters instead of regex escape strings.

In [ ]:
let operatorRule = """| ['+' '-' '*' '/' '(' ')'] { Symbol((text lexbuf).[0]) }"""
printfn "%s" operatorRule

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
let numberRule = """| natural { Number(int (text lexbuf)) }"""
printfn "%s" numberRule

The rule below is used to keep track of line numbers. We use the function `len` since there might be
more than one newline.  The member variable `lexer.lineno` keeps track of the current line number.  This variable
is maintained so that we are able to specify the precise location of unkown characters in error messages.

In [ ]:
let newlineRule = """| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }"""
printfn "%s" newlineRule

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
let whitespaceRule = """| [' ' '\t' '\r']+ { token lexbuf }"""
printfn "%s" whitespaceRule

FsLex rules pair a pattern with a typed F# action. The complete `.fsl` specification is compiled by the adjacent project; the matching rule is shown below. Scanner state is represented by mutually recursive rules and source positions by `LexBuffer<char>`. Unlike PLY, FsLex needs no notebook filename workaround. Longest matches win; ties use rule order.

In [ ]:
let errorRule = """| _ { Illegal((text lexbuf).[0]) }"""
printfn "%s" errorRule

Build `Scanning.fsproj` before opening this notebook. Generation happens at build time, so no filename workaround is required.

In [ ]:
let scan = Scanning.arithmetic

`Scanning.arithmetic` runs the generated scanner with a fresh lexing buffer on each call.

In [ ]:
File.ReadAllText("Scanning/Arithmetic.fsl") |> printfn "%s"

Lets test the generated scanner, that is stored in `lexer`, with the following string:

In [ ]:
let data = """
       3 + 4 * 10 + 007 + (-20) * 2
       42
       a
       """

The immutable input is passed to a fresh scanner buffer.

In [ ]:
printfn "%s" data

Every scan initializes line and offset information; repeated execution gives the same locations.

In [ ]:
let tokens = scan data

We now iterate over the typed token list, reporting illegal characters while retaining the remaining tokens.

In [ ]:
for token in tokens do
    match token.Value with
    | Illegal c -> printfn "Illegal character %c at line %d, offset %d" c token.Line token.Offset
    | _ -> printfn "%A" token

We see that the generated tokens contain four pieces of information:
 1. The *type* of the token.
 2. The *value* of the token.  This is either a number or a string.
 3. The *line number* of the token.  The line number starts with 1.
    However, note that the first line of `data` is empty.
 4. The *character count*.  For example, the last token is the $54^{\textrm{th}}$ character.
    The character count starts with `0`.